# C.6 正確答案怎麼變成訓練回饋？

# C：推理步驟、驗證與測試時算力

前置：下一字生成、SFT與標準答案驗證。像解數學題：寫出步驟、多試幾種辦法、檢查答案，是三個不同動作。真實推理質量需要trained模型；默認例子用可驗證的數字，避免把長文字當思考保證。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：多試幾次，還要知道怎麼選**

候選包含正解，不代表最後答案已經選中。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/reasoning.svg")))

**先想一想：**只獎勵有很多步驟，能保證答案對嗎？

verifier給可驗證reward，例如答案正確1、不正確0。風格偏好則來自另一種判斷；reward定義決定模型被鼓勵做什麼。

**把直覺寫成數學：**r_correct(answer)=1[answer=truth]；嚴格parser與無答案狀態也要定義。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
answers = [3, 4, 5]
truth = 4
rewards = [float(a == truth) for a in answers]
print(list(zip(answers, rewards)))
assert rewards == [0.0, 1.0, 0.0]

**如何讀結果：**這只是reward生成器；還沒更新policy。資料泄漏或答案格式漏洞會讓reward失效。

**只改一件事：**只換truth，保持候選不變。
